# 🍅 Tomato MTL — Ablation Study Evaluation
**Purpose:** Load saved `.pth` models + dataset → compute & save all ablation results

| Model File | Description |
|---|---|
| `best_resnet50.pth` | Full MTL model (CBAM + Cross-Task Attention) |
| `single_task_best.pth` | Single-Task baseline (Disease-only head) |
| `last_weight_model.pth` | Last loss-weight sensitivity model |

**Outputs saved:** `ablation_mtl_vs_singletask.xlsx`, `ablation_loss_weights.xlsx`, `ablation_backbone_comparison.xlsx`, `final_results_summary.xlsx` + confusion matrix plots → zipped for download

## ⚙️ Step 1 — Install Dependencies

In [ ]:
!pip install -q timm albumentations openpyxl

import torch, os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import torch.nn as nn
import torch.nn.functional as F
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMG_SIZE = 224

DISEASE_CLASSES  = ['Bacterial_spot','Early_blight','Late_blight','Leaf_Mold','Septoria_leaf_spot','healthy']
SEVERITY_CLASSES = ['Early','Mid','Late']

NUM_DISEASE = len(DISEASE_CLASSES)
NUM_SEVERITY = len(SEVERITY_CLASSES)

print("✅ Setup done | Device:", DEVICE)

✅ Setup done | Device: cuda


In [ ]:

# 🔹 Upload kaggle.json
from google.colab import files
print("⬆️ Upload kaggle.json")
files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# 🔹 Download dataset
!kaggle datasets download -d janiruwalisingha/tomato-leaf-disease-severity-dataset \
       --path /content/data --unzip

print("✅ Dataset downloaded")


# 🔹 Model paths (already uploaded)
MTL_PATH = "/content/best_resnet50.pth"
LW_PATH  = "/content/last_weight_model.pth"
ST_PATH  = "/content/single_task_best.pth"

⬆️ Upload kaggle.json


Saving kaggle.json to kaggle (2).json
Dataset URL: https://www.kaggle.com/datasets/janiruwalisingha/tomato-leaf-disease-severity-dataset
License(s): unknown
100% 1.08G/1.08G [00:15<00:00, 74.5MB/s]

✅ Dataset downloaded


In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from torch.utils.data import Dataset, DataLoader

DISEASE_VAL  = '/content/data/disease/val'
SEVERITY_VAL = '/content/data/severity/valid'

def get_tf():
    return A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=[0.485,0.456,0.406],
                    std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])

SEV_DIS_CLASSES = ['Bacterial_spot','Early_blight','Late_blight','Leaf_Mold','Septoria_leaf_spot']

class MTLDataset(Dataset):
    def __init__(self):
        self.samples = []
        self.tf = get_tf()

        # Disease
        for cls in DISEASE_CLASSES:
            d = os.path.join(DISEASE_VAL, cls)
            if not os.path.isdir(d): continue
            for f in os.listdir(d):
                if f.endswith(('.jpg','.png','.jpeg')):
                    self.samples.append((os.path.join(d,f), DISEASE_CLASSES.index(cls), -1, False))

        # Severity
        for dis in SEV_DIS_CLASSES:
            for sev in SEVERITY_CLASSES:
                d = os.path.join(SEVERITY_VAL, dis, sev)
                if not os.path.isdir(d): continue
                for f in os.listdir(d):
                    if f.endswith(('.jpg','.png','.jpeg')):
                        self.samples.append((os.path.join(d,f),
                                             DISEASE_CLASSES.index(dis),
                                             SEVERITY_CLASSES.index(sev),
                                             True))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        p, d, s, h = self.samples[idx]
        img = np.array(Image.open(p).convert('RGB'))
        img = self.tf(image=img)['image']
        return img, torch.tensor(d), torch.tensor(s), torch.tensor(h)

val_loader = DataLoader(MTLDataset(), batch_size=32, shuffle=False)

print("✅ Validation loader ready:", len(val_loader))

✅ Validation loader ready: 23


In [ ]:
# ─── CBAM ─────────────────────────────────────

class ChannelAttention(nn.Module):
    def __init__(self, c, r=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.max = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(c, c//r, bias=False),   # FIXED
            nn.ReLU(),
            nn.Linear(c//r, c, bias=False)    # FIXED
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        return x * torch.sigmoid(
            self.fc(self.avg(x).view(b,c)) +
            self.fc(self.max(x).view(b,c))
        ).view(b,c,1,1)


class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)  # FIXED

    def forward(self, x):
        return x * torch.sigmoid(
            self.conv(torch.cat([x.mean(1,True), x.max(1,True)[0]], 1))
        )


class CBAM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ca = ChannelAttention(c)
        self.sa = SpatialAttention()

    def forward(self, x):
        return self.sa(self.ca(x))


# ─── Cross Task Attention ─────────────────────

class CrossTaskAttention(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.q = nn.Linear(d, d)
        self.k = nn.Linear(d, d)
        self.v = nn.Linear(d, d)
        self.norm = nn.LayerNorm(d)
        self.scale = d ** -0.5

    def forward(self, sf, df):
        attn = torch.sigmoid(
            (self.q(sf) * self.k(df)).sum(-1, keepdim=True) * self.scale
        )
        return self.norm(sf + attn * self.v(df))


# ─── Task Head ────────────────────────────────

class TaskHead(nn.Module):
    def __init__(self, in_f, h, n):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_f, h),
            nn.BatchNorm1d(h),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(h, n)
        )

    def forward(self, x):
        return self.net(x)


# ─── MAIN MTL MODEL ───────────────────────────

class TomatoMTL(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            'resnet50',
            pretrained=False,
            num_classes=0,
            global_pool=''
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        with torch.no_grad():
            fd = self.backbone(torch.zeros(2,3,IMG_SIZE,IMG_SIZE)).shape[1]

        self.cbam = CBAM(fd)

        self.dis_proj = nn.Sequential(
            nn.Linear(fd,512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4)
        )
        self.dis_head = TaskHead(512,256,NUM_DISEASE)

        self.sev_proj = nn.Sequential(
            nn.Linear(fd,512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4)
        )
        self.cross_attn = CrossTaskAttention(512)
        self.sev_head = TaskHead(512,256,NUM_SEVERITY)

    def forward(self, x):
        f = self.backbone(x)

        if f.dim() == 4:
            f = self.cbam(f)
            f = self.pool(f).flatten(1)

        df = self.dis_proj(f)
        dl = self.dis_head(df)

        sf = self.sev_proj(f)
        sf = self.cross_attn(sf, df.detach())   # FIXED NAME
        sl = self.sev_head(sf)

        return dl, sl


# ─── SINGLE TASK MODEL ────────────────────────

class SingleTaskModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.backbone = timm.create_model(
            'resnet50',
            pretrained=False,
            num_classes=0,
            global_pool=''
        )

        self.pool = nn.AdaptiveAvgPool2d(1)

        with torch.no_grad():
            fd = self.backbone(torch.zeros(2,3,IMG_SIZE,IMG_SIZE)).shape[1]

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fd,512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, NUM_DISEASE)
        )

    def forward(self, x):
        f = self.backbone(x)

        if f.dim() == 4:
            f = self.pool(f)

        return self.head(f), torch.zeros(x.size(0), NUM_SEVERITY).to(x.device)

print("✅ Model ready (fully corrected)")


✅ Model ready (fully corrected)


In [ ]:
LW_PATH  = "/content/last_weight_model.pth"
ST_PATH  = "/content/single_task_best.pth"

# Loss-weight MTL
lw_model = TomatoMTL().to(DEVICE)
lw_model.load_state_dict(torch.load(LW_PATH, map_location=DEVICE), strict=True)
lw_model.eval()

# Single-task
st_model = SingleTaskModel().to(DEVICE)
st_model.load_state_dict(torch.load(ST_PATH, map_location=DEVICE), strict=True)
st_model.eval()

print("✅ Loaded: Loss-weight + Single-task")

✅ Loaded: Loss-weight + Single-task


In [ ]:
def full_eval(model, loader):
    model.eval()

    dt, dp, dprob = [], [], []
    st, sp, sprob = [], [], []

    with torch.no_grad():
        for imgs, dis_l, sev_l, has_sev in loader:
            imgs = imgs.to(DEVICE)

            dl, sl = model(imgs)

            probs_d = torch.softmax(dl, dim=1)
            probs_s = torch.softmax(sl, dim=1)

            dt.extend(dis_l.numpy())
            dp.extend(probs_d.argmax(1).cpu().numpy())
            dprob.extend(probs_d.cpu().numpy())

            if has_sev.any():
                idx = has_sev.bool()
                st.extend(sev_l[idx].numpy())
                sp.extend(probs_s[idx].argmax(1).cpu().numpy())
                sprob.extend(probs_s[idx].cpu().numpy())

    return dt, dp, np.array(dprob), st, sp, np.array(sprob)

In [ ]:
def evaluate_full_style(model, name):
    dt, dp, _, st, sp, _ = full_eval(model, val_loader)

    return {
        "Model": name,

        # Disease metrics
        "Disease Acc (%)": round(accuracy_score(dt, dp) * 100, 2),
        "Disease F1": round(f1_score(dt, dp, average='weighted', zero_division=0), 4),
        "Disease Precision": round(precision_score(dt, dp, average='weighted', zero_division=0), 4),
        "Disease Recall": round(recall_score(dt, dp, average='weighted', zero_division=0), 4),

        # Severity metrics
        "Severity Acc (%)": round(accuracy_score(st, sp) * 100, 2),
        "Severity F1": round(f1_score(st, sp, average='weighted', zero_division=0), 4),
        "Severity Precision": round(precision_score(st, sp, average='weighted', zero_division=0), 4),
        "Severity Recall": round(recall_score(st, sp, average='weighted', zero_division=0), 4),
    }


# ─── Evaluate only 2 models ─────────────────────────
results = [
    evaluate_full_style(lw_model, "MTL (Loss Weight)"),
    evaluate_full_style(st_model, "Single Task")
]

df = pd.DataFrame(results)

print("\n📊 FINAL ABLATION (SAME AS TRAINING)")
print(df.to_string(index=False))

# Save results
save_path = "/content/ablation_2models_final.xlsx"
df.to_excel(save_path, index=False)

print(f"\n✅ Saved: {save_path}")


📊 FINAL ABLATION (SAME AS TRAINING)
            Model  Disease Acc (%)  Disease F1  Disease Precision  Disease Recall  Severity Acc (%)  Severity F1  Severity Precision  Severity Recall
MTL (Loss Weight)            82.60      0.8276             0.8375          0.8260             75.38       0.7488              0.7551           0.7538
      Single Task            92.74      0.9274             0.9277          0.9274             49.23       0.3248              0.2424           0.4923

✅ Saved: /content/ablation_2models_final.xlsx


---
## 📋 Summary of Outputs

| File | Contents |
|------|----------|
| `results/ablation_mtl_vs_singletask.xlsx` | MTL vs Single-Task comparison |
| `results/ablation_loss_weights.xlsx` | Loss weight sensitivity (w=0.6/0.4 vs w=0.8/0.2) |
| `results/final_results_summary.xlsx` | Best MTL model metrics |
| `plots/cm_disease_resnet50_mtl.png` | Disease confusion matrix (MTL) |
| `plots/cm_severity_resnet50_mtl.png` | Severity confusion matrix (MTL) |
| `plots/cm_disease_resnet50_singletask.png` | Disease confusion matrix (Single-Task) |